Cell 1: Environment & GPU Check

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

!pip install -qqq seqeval

from seqeval.metrics import classification_report as seqeval_classification_report

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("Device Name:", torch.cuda.get_device_name(0))
else:
  print(
      "⚠️ WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4"
      " GPU."
  )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
PyTorch Version: 2.11.0+cu128
GPU Available: True
Device Name: Tesla T4


Fixed some logical issues, re-checked the weights, added early stopping

In [ ]:
# ============================================================
# 1. MOUNT GOOGLE DRIVE & SETUP DIRECTORIES
# ============================================================
import os
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from seqeval.metrics import f1_score as seqeval_f1_score
from google.colab import drive
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

# Mount Google Drive
drive.mount('/content/drive')

MODEL_NAME = "dicta-il/dictabert"
DRIVE_SAVE_PATH = "/content/drive/MyDrive/WORKSHOP/Stage_1/models/stage1_dictabert_fix1/"
TMP_SEARCH_PATH = "/tmp/stage1_weight_search/"

os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
os.makedirs(TMP_SEARCH_PATH, exist_ok=True)

LABEL_2_ID = {"O": 0, "B-MU": 1, "I-MU": 2}
ID_2_LABEL = {0: "O", 1: "B-MU", 2: "I-MU"}

# ============================================================
# 2. PRODUCTION DATASET CLASS WITH TRUNCATION DIAGNOSTICS
# ============================================================
class GroupedStage1BIODataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_len=128):
        self.tokenizer = tokenizer
        self.max_len = max_len

        df = pd.read_csv(csv_path)
        df = df.dropna(subset=["Full_Raw_Sentence", "MeaningVal"])

        self.grouped_data = []
        total_mvs = 0
        matched_mvs = 0

        # Group by sentence preserving order
        for sent, group in df.groupby("Full_Raw_Sentence", sort=False):
            sentence = str(sent).strip()
            mv_spans = []

            # Search pointer prevents matching duplicate words to index 0
            search_start_pos = 0

            for _, row in group.iterrows():
                mv = str(row["MeaningVal"]).strip()
                if not mv:
                    continue
                total_mvs += 1

                # Search sequentially from where previous MV ended
                start_pos = sentence.find(mv, search_start_pos)
                if start_pos == -1:
                    start_pos = sentence.find(mv)

                if start_pos != -1:
                    end_pos = start_pos + len(mv)
                    mv_spans.append({"text": mv, "start": start_pos, "end": end_pos})
                    matched_mvs += 1
                    search_start_pos = end_pos

            if mv_spans:
                self.grouped_data.append(
                    {"sentence": sentence, "meaning_spans": mv_spans}
                )

        print(
            f"📦 Loaded {len(self.grouped_data)} unique aggregated sentences from"
            f" {csv_path} | Matched {matched_mvs}/{total_mvs} MeaningValues"
        )

        # Truncation Check
        truncated_count = 0
        max_tokens_found = 0

        for item in self.grouped_data:
            raw_tokens = self.tokenizer.encode(item["sentence"], add_special_tokens=True)
            token_len = len(raw_tokens)
            if token_len > max_tokens_found:
                max_tokens_found = token_len
            if token_len > self.max_len:
                truncated_count += 1

        print(f"📊 Truncation Summary for max_len={self.max_len}:")
        print(f"   • Longest sentence in dataset : {max_tokens_found} tokens")
        if truncated_count > 0:
            print(f"   • ⚠️ WARNING: {truncated_count}/{len(self.grouped_data)} sentences ({truncated_count/len(self.grouped_data)*100:.1f}%) were TRUNCATED!")
        else:
            print(f"   • ✅ Perfect coverage: 0 sentences were truncated.")

    def __len__(self):
        return len(self.grouped_data)

    def __getitem__(self, idx):
        item = self.grouped_data[idx]
        sentence = item["sentence"]
        meaning_spans = item["meaning_spans"]

        encoding = self.tokenizer(
            sentence,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        offsets = encoding["offset_mapping"].squeeze(0).tolist()
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # Detect character limit reached by tokenizer
        valid_offsets = [o for o in offsets if o != [0, 0]]
        last_covered_char = valid_offsets[-1][1] if valid_offsets else 0
        is_truncated = last_covered_char < len(sentence)

        labels = [LABEL_2_ID["O"]] * len(offsets)

        # Tag special tokens with -100
        for i, (start_tok, end_tok) in enumerate(offsets):
            if start_tok == 0 and end_tok == 0:
                labels[i] = -100

        sorted_spans = sorted(meaning_spans, key=lambda x: x["start"])

        for span in sorted_spans:
            mv_start = span["start"]
            mv_end = span["end"]

            if is_truncated and mv_start >= last_covered_char:
                continue

            first_token_matched = False

            for idx_tok, (start_tok, end_tok) in enumerate(offsets):
                if labels[idx_tok] == -100:
                    continue

                if not (end_tok <= mv_start or start_tok >= mv_end):
                    if not first_token_matched:
                        labels[idx_tok] = LABEL_2_ID["B-MU"]
                        first_token_matched = True
                    else:
                        labels[idx_tok] = LABEL_2_ID["I-MU"]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(labels, dtype=torch.long),
        }

# ============================================================
# 3. TRAINER & METRICS DEFINITION
# ============================================================
class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        weights = (
            torch.tensor(self.class_weights, dtype=torch.float).to(logits.device)
            if self.class_weights
            else None
        )
        loss_fct = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]

    preds = np.argmax(logits, axis=2)

    true_preds, true_labs = [], []
    for p_row, l_row in zip(preds, labels):
        c_preds, c_labs = [], []
        for p, l in zip(p_row, l_row):
            if l != -100:
                c_preds.append(ID_2_LABEL.get(int(p), "O"))
                c_labs.append(ID_2_LABEL.get(int(l), "O"))
        true_preds.append(c_preds)
        true_labs.append(c_labs)

    span_f1 = seqeval_f1_score(true_labs, true_preds)
    return {"span_f1": span_f1}

# ============================================================
# 4. LOAD TOKENIZER & DATASETS
# ============================================================
print("\n📦 Loading Tokenizer and Datasets...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

TRAIN_CSV = "train_set.csv" if os.path.exists("train_set.csv") else "stage1_splits/train_set.csv"
VAL_CSV = "val_set.csv" if os.path.exists("val_set.csv") else "stage1_splits/val_set.csv"

train_dataset = GroupedStage1BIODataset(TRAIN_CSV, tokenizer)
val_dataset = GroupedStage1BIODataset(VAL_CSV, tokenizer)

# ============================================================
# 5. WEIGHT GRID SEARCH (WITH EARLY STOPPING)
# ============================================================
weight_candidates = [
    [1.0, 1.0, 1.0],  # Unweighted baseline
    [1.0, 3.0, 2.0],  # Balanced boost
    [1.0, 5.0, 4.0],  # Aggressive boost
]

best_f1 = 0.0
best_weights = None

print("\n🔍 Starting Automated Loss Weight Search on Clean Dataset...")

for idx, weights in enumerate(weight_candidates):
    print(f"\n🧪 Trial {idx+1}/{len(weight_candidates)} — Testing Weights [O, B-MU, I-MU]: {weights}")

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME, num_labels=3, id2label=ID_2_LABEL, label2id=LABEL_2_ID
    )

    search_args = TrainingArguments(
        output_dir=os.path.join(TMP_SEARCH_PATH, f"trial_{idx}"),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=3e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        logging_steps=20,
        load_best_model_at_end=True,
        metric_for_best_model="span_f1",
        greater_is_better=True,
        save_total_limit=1,
        report_to="none",
    )

    trainer = WeightedLossTrainer(
        model=model,
        args=search_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        class_weights=weights,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()

    eval_res = trainer.evaluate()
    f1 = eval_res["eval_span_f1"]
    print(f"🎯 Validation Span-Level F1 for {weights}: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_weights = weights

print("\n" + "=" * 65)
print(f"🏆 WINNING WEIGHT CONFIGURATION: {best_weights} (Validation Span F1: {best_f1:.4f})")
print("=" * 65)

# ============================================================
# 6. FINAL RUN & SAVE TO GOOGLE DRIVE
# ============================================================
print(f"\n🚀 Training Final Production Model for 8 Epochs using Weights {best_weights}...")

final_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=ID_2_LABEL, label2id=LABEL_2_ID
)

production_args = TrainingArguments(
    output_dir=os.path.join(DRIVE_SAVE_PATH, "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="span_f1",
    greater_is_better=True,
    save_total_limit=1,
)

final_trainer = WeightedLossTrainer(
    model=final_model,
    args=production_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    class_weights=best_weights,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

final_trainer.train()

final_model.save_pretrained(DRIVE_SAVE_PATH)
tokenizer.save_pretrained(DRIVE_SAVE_PATH)

print("\n" + "=" * 65)
print(f"🎉 FINAL STAGE 1 MODEL SUCCESSFULLY TRAINED & EXPORTED TO:")
print(f" 📁 {DRIVE_SAVE_PATH}")
print("=" * 65)

Mounted at /content/drive

📦 Loading Tokenizer and Datasets...


config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: reconstructing file:   0%|          |  0.00B / 1.50MB            

vocab.txt: downloading bytes:           |  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🎉 FINAL STAGE 1 MODEL SUCCESSFULLY TRAINED & EXPORTED TO:
 📁 /content/drive/MyDrive/WORKSHOP/Stage_1/models/stage1_dictabert_fix1/


In [ ]:
# ============================================================
# INSPECT METRICS & PREDICTIONS
# ============================================================
import os
import numpy as np
import pandas as pd
import torch
from seqeval.metrics import classification_report as seqeval_classification_report
from transformers import AutoModelForTokenClassification, AutoTokenizer

DRIVE_SAVE_PATH = "/content/drive/MyDrive/WORKSHOP/Stage_1/models/stage1_dictabert_fix1/"
VAL_CSV = "val_set.csv" if os.path.exists("val_set.csv") else "stage1_splits/val_set.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"📦 Loading Saved Stage 1 Model from: {DRIVE_SAVE_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(DRIVE_SAVE_PATH)
model = AutoModelForTokenClassification.from_pretrained(DRIVE_SAVE_PATH).to(device)
model.eval()

# Reload Dataset
val_dataset = GroupedStage1BIODataset(VAL_CSV, tokenizer)

# ------------------------------------------------------------
# 1. COMPUTE FULL REPORT
# ------------------------------------------------------------
print("\n📊 Evaluating Final Saved Model on Validation Set...")

# Create evaluation dataloader
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False)

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].numpy()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.cpu().numpy()
        preds = np.argmax(logits, axis=2)

        for p_row, l_row in zip(preds, labels):
            c_preds, c_labs = [], []
            for p, l in zip(p_row, l_row):
                if l != -100:
                    c_preds.append(ID_2_LABEL[p])
                    c_labs.append(ID_2_LABEL[l])
            all_preds.append(c_preds)
            all_labels.append(c_labs)

print("\n" + "=" * 65)
print("🏆 FINAL STAGE 1 SPAN-LEVEL CLASSIFICATION REPORT")
print("=" * 65)
print(seqeval_classification_report(all_labels, all_preds, digits=4))
print("=" * 65)

# ------------------------------------------------------------
# 2. QUALITATIVE INSPECTION
# ------------------------------------------------------------
def extract_spans_from_bio(token_ids, pred_ids, offsets, sentence, tokenizer):
    spans = []
    current_tokens = []

    for token_id, p_id, (start_char, end_char) in zip(token_ids, pred_ids, offsets):
        token = tokenizer.convert_ids_to_tokens(int(token_id))
        if token in tokenizer.all_special_tokens or (start_char == 0 and end_char == 0):
            if current_tokens:
                spans.append(current_tokens)
                current_tokens = []
            continue

        if p_id == 1:  # B-MU
            if current_tokens:
                spans.append(current_tokens)
            current_tokens = [(start_char, end_char)]
        elif p_id == 2:  # I-MU
            if current_tokens:
                current_tokens.append((start_char, end_char))
            else:
                current_tokens = [(start_char, end_char)]
        else:  # O
            if current_tokens:
                spans.append(current_tokens)
                current_tokens = []

    if current_tokens:
        spans.append(current_tokens)

    extracted_mvs = []
    for span in spans:
        min_start = span[0][0]
        max_end = span[-1][1]
        phrase = sentence[min_start:max_end].strip()
        if phrase:
            extracted_mvs.append(phrase)
    return extracted_mvs

val_df = pd.read_csv(VAL_CSV)
unique_val_sentences = val_df["Full_Raw_Sentence"].dropna().unique()

print("\n🔍 Inspecting 5 Validation Predictions (Side-by-Side):\n")
for i in range(min(5, len(unique_val_sentences))):
    sentence = str(unique_val_sentences[i])
    target_mvs = val_df[val_df["Full_Raw_Sentence"] == sentence]["MeaningVal"].dropna().tolist()

    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        return_offsets_mapping=True,
    ).to(device)

    offsets = inputs.pop("offset_mapping")[0].cpu().tolist()

    with torch.no_grad():
        outputs = model(**inputs)

    pred_ids = torch.argmax(outputs.logits, dim=2)[0].cpu().tolist()
    token_ids = inputs["input_ids"][0].cpu().tolist()

    predicted_mvs = extract_spans_from_bio(token_ids, pred_ids, offsets, sentence, tokenizer)

    print(f"Sample {i+1}:")
    print(f"  Sentence            : {sentence}")
    print(f"  Target MeaningVals  : {target_mvs}")
    print(f"  Model Predicted MVs : {predicted_mvs}")
    print("-" * 65)

📦 Loading Saved Stage 1 Model from: /content/drive/MyDrive/WORKSHOP/Stage_1/models/stage1_dictabert_fix1/...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

📦 Loaded 796 unique aggregated sentences from val_set.csv | Matched 4100/4569 MeaningValues
📊 Truncation Summary for max_len=128:
   • Longest sentence in dataset : 85 tokens
   • ✅ Perfect coverage: 0 sentences were truncated.

📊 Evaluating Final Saved Model on Validation Set...

🏆 FINAL STAGE 1 SPAN-LEVEL CLASSIFICATION REPORT
              precision    recall  f1-score   support

          MU     0.7258    0.7817    0.7527      4036

   micro avg     0.7258    0.7817    0.7527      4036
   macro avg     0.7258    0.7817    0.7527      4036
weighted avg     0.7258    0.7817    0.7527      4036


🔍 Inspecting 5 Validation Predictions (Side-by-Side):

Sample 1:
  Sentence            : הכתובת שאני שמה בוויז כדי להגיע לבית
  Target MeaningVals  : ['כתובת', 'שאני ', 'שמה', 'בוויז', 'כדי להגיע', 'לבית']
  Model Predicted MVs : ['הכתובת', 'שאני', 'שמה', 'בוויז', 'כדי להגיע', 'לבית']
-----------------------------------------------------------------
Sample 2:
  Sentence            : סוג של כל

EVALUATION (still on val)

In [ ]:
# ============================================================
# DETAILED MODEL 1 EVALUATION
# ============================================================
import os
import numpy as np
import pandas as pd
import torch
from collections import Counter
from seqeval.metrics import classification_report as seqeval_classification_report
from sklearn.metrics import classification_report as sklearn_classification_report
from transformers import AutoModelForTokenClassification, AutoTokenizer

DRIVE_SAVE_PATH = "/content/drive/MyDrive/WORKSHOP/Stage_1/models/stage1_dictabert_fix1/"
VAL_CSV = "val_set.csv" if os.path.exists("val_set.csv") else "stage1_splits/val_set.csv"

ID_2_LABEL = {0: "O", 1: "B-MU", 2: "I-MU"}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"📦 Loading Model 1 from: {DRIVE_SAVE_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(DRIVE_SAVE_PATH)
model = AutoModelForTokenClassification.from_pretrained(DRIVE_SAVE_PATH).to(device)
model.eval()

# ------------------------------------------------------------
# 1. HELPER FUNCTIONS FOR EXTRACTING SPANS AND COMPUTING IOU
# ------------------------------------------------------------
def normalize_hebrew(text):
    text = str(text)
    import re
    text = re.sub(r"[^\w\s]", "", text)
    return " ".join(text.strip().split())

def compute_word_iou(pred_phrase, gold_phrase):
    pred_words = normalize_hebrew(pred_phrase).split()
    gold_words = normalize_hebrew(gold_phrase).split()

    if not pred_words or not gold_words:
        return 1.0 if pred_words == gold_words else 0.0

    p_counter, g_counter = Counter(pred_words), Counter(gold_words)
    intersection = sum((p_counter & g_counter).values())
    union = sum((p_counter | g_counter).values())

    return intersection / union if union > 0 else 0.0

def extract_spans_from_bio(token_ids, pred_ids, offsets, sentence, tokenizer):
    spans = []
    current_tokens = []

    for token_id, p_id, (start_char, end_char) in zip(token_ids, pred_ids, offsets):
        token = tokenizer.convert_ids_to_tokens(int(token_id))
        if token in tokenizer.all_special_tokens or (start_char == 0 and end_char == 0):
            if current_tokens:
                spans.append(current_tokens)
                current_tokens = []
            continue

        if p_id == 1:  # B-MU
            if current_tokens:
                spans.append(current_tokens)
            current_tokens = [(start_char, end_char)]
        elif p_id == 2:  # I-MU
            if current_tokens:
                current_tokens.append((start_char, end_char))
            else:
                current_tokens = [(start_char, end_char)]
        else:  # O
            if current_tokens:
                spans.append(current_tokens)
                current_tokens = []

    if current_tokens:
        spans.append(current_tokens)

    extracted_mvs = []
    for span in spans:
        min_start = span[0][0]
        max_end = span[-1][1]
        phrase = sentence[min_start:max_end].strip()
        phrase_clean = normalize_hebrew(phrase)
        if phrase_clean:
            extracted_mvs.append(phrase_clean)
    return extracted_mvs

# ------------------------------------------------------------
# 2. RUN TOKEN & SPAN EVALUATION
# ------------------------------------------------------------
val_dataset = GroupedStage1BIODataset(VAL_CSV, tokenizer)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False)

flat_token_preds = []
flat_token_labels = []
seqeval_preds = []
seqeval_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].numpy()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = np.argmax(outputs.logits.cpu().numpy(), axis=2)

        for p_row, l_row in zip(preds, labels):
            c_preds, c_labs = [], []
            for p, l in zip(p_row, l_row):
                if l != -100:
                    flat_token_preds.append(p)
                    flat_token_labels.append(l)
                    c_preds.append(ID_2_LABEL[p])
                    c_labs.append(ID_2_LABEL[l])
            seqeval_preds.append(c_preds)
            seqeval_labels.append(c_labs)

print("\n" + "=" * 65)
print("1️⃣ TOKEN-LEVEL CLASSIFICATION REPORT (O vs. B-MU vs. I-MU)")
print("=" * 65)
print(sklearn_classification_report(
    flat_token_labels,
    flat_token_preds,
    target_names=["O", "B-MU", "I-MU"],
    digits=4
))

print("\n" + "=" * 65)
print("2️⃣ STRICT SPAN-LEVEL REPORT (SEQEVAL EXACT PHRASE MATCH)")
print("=" * 65)
print(seqeval_classification_report(seqeval_labels, seqeval_preds, digits=4))

# ------------------------------------------------------------
# 3. RUN IOU & SENTENCE EXACT MATCH EVALUATION
# ------------------------------------------------------------
val_df = pd.read_csv(VAL_CSV)
grouped = val_df.groupby("Full_Raw_Sentence", sort=False)

sentence_em_correct = 0
total_sentences = 0
all_span_ious = []

for sentence_raw, group in grouped:
    sentence = str(sentence_raw).strip()
    gold_mvs = [normalize_hebrew(x) for x in group["MeaningVal"].dropna().astype(str).tolist()]

    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        return_offsets_mapping=True,
    ).to(device)

    offsets = inputs.pop("offset_mapping")[0].cpu().tolist()

    with torch.no_grad():
        outputs = model(**inputs)

    pred_ids = torch.argmax(outputs.logits, dim=2)[0].cpu().tolist()
    token_ids = inputs["input_ids"][0].cpu().tolist()

    pred_mvs = extract_spans_from_bio(token_ids, pred_ids, offsets, sentence, tokenizer)

    # Calculate Sentence-Level Exact Set Match
    if set(pred_mvs) == set(gold_mvs):
        sentence_em_correct += 1
    total_sentences += 1

    # Calculate Word IoU per Gold Meaning Value
    for gold_mv in gold_mvs:
        best_mv_iou = 0.0
        for pred_mv in pred_mvs:
            iou = compute_word_iou(pred_mv, gold_mv)
            if iou > best_mv_iou:
                best_mv_iou = iou
        all_span_ious.append(best_mv_iou)

avg_iou = np.mean(all_span_ious) * 100 if all_span_ious else 0.0
sentence_em_acc = (sentence_em_correct / max(total_sentences, 1)) * 100

print("=" * 65)
print("3️⃣ ADVANCED OVERLAP METRICS (IOU & SENTENCE MATCH)")
print("=" * 65)
print(f" • Mean Phrase Word-Level IoU : {avg_iou:.2f}%")
print(f"   (Measures how much text overlap predicted spans share with ground truth)")
print(f" • Sentence Exact Set Match   : {sentence_em_acc:.2f}% ({sentence_em_correct}/{total_sentences})")
print(f"   (Strict binary metric used in the cascade script)")
print("=" * 65)

📦 Loading Model 1 from: /content/drive/MyDrive/WORKSHOP/Stage_1/models/stage1_dictabert_fix1/...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

📦 Loaded 796 unique aggregated sentences from val_set.csv | Matched 4100/4569 MeaningValues
📊 Truncation Summary for max_len=128:
   • Longest sentence in dataset : 85 tokens
   • ✅ Perfect coverage: 0 sentences were truncated.

1️⃣ TOKEN-LEVEL CLASSIFICATION REPORT (O vs. B-MU vs. I-MU)
              precision    recall  f1-score   support

           O     0.8261    0.7289    0.7745      2855
        B-MU     0.8386    0.8511    0.8448      4036
        I-MU     0.7587    0.8459    0.7999      2401

    accuracy                         0.8122      9292
   macro avg     0.8078    0.8086    0.8064      9292
weighted avg     0.8141    0.8122    0.8116      9292


2️⃣ STRICT SPAN-LEVEL REPORT (SEQEVAL EXACT PHRASE MATCH)
              precision    recall  f1-score   support

          MU     0.7258    0.7817    0.7527      4036

   micro avg     0.7258    0.7817    0.7527      4036
   macro avg     0.7258    0.7817    0.7527      4036
weighted avg     0.7258    0.7817    0.7527      4036